# Landing to Bronze
### CineData Analytics
### RocketLab 2026.2

In [0]:
# Configurações do ambiente

catalog = "rocketlab"
bronze_schema_name = "bronze"

bronze_schema = f"{catalog}.{bronze_schema_name}"

# Volume que contém os arquivos brutos
landing_path = f"/Volumes/{catalog}/default/inputs"

print(f"Catalog: {catalog}")
print(f"Bronze schema: {bronze_schema}")
print(f"Landing path: {landing_path}")

Catalog: rocketlab
Bronze schema: rocketlab.bronze
Landing path: /Volumes/rocketlab/default/inputs


In [0]:
# Criação do schema Bronze

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {bronze_schema}"
)

print(f"Schema {bronze_schema} disponível.")

display(
    spark.sql(f"SHOW SCHEMAS IN {catalog}")
)

Schema rocketlab.bronze disponível.


databaseName
bronze
default
gold
information_schema
silver


## Validação dos arquivos de entrada

Antes de iniciar a ingestão, é verificada a presença dos cinco arquivos CSV esperados no Volume de entrada.



In [0]:
expected_files = [
    "movies_info_TMDB_IMDB.csv",
    "movies_financials_IMDB_TMDB.csv",
    "movies_metrics_IMDB_TMDB.csv",
    "credits_and_tags_IMDB_TMDB.csv",
    "movies_reviews.csv"
]

existing_files = {
    file.name for file in dbutils.fs.ls(landing_path)
}

missing_files = [
    file for file in expected_files
    if file not in existing_files
]

if missing_files:
    print("Arquivos não encontrados:")

    for file in missing_files:
        print(f"  - {file}")

else:
    print("Todos os arquivos esperados foram encontrados.")

    for file in expected_files:
        print(f"  - {file}")


Todos os arquivos esperados foram encontrados.
  - movies_info_TMDB_IMDB.csv
  - movies_financials_IMDB_TMDB.csv
  - movies_metrics_IMDB_TMDB.csv
  - credits_and_tags_IMDB_TMDB.csv
  - movies_reviews.csv


## Mapeamento dos arquivos para as tabelas Bronze

In [0]:
files_to_tables = {
    "movies_info_TMDB_IMDB.csv": "tb_movies_info",
    "movies_financials_IMDB_TMDB.csv": "tb_movies_financials",
    "movies_metrics_IMDB_TMDB.csv": "tb_movies_metrics",
    "credits_and_tags_IMDB_TMDB.csv": "tb_credits_and_tags",
    "movies_reviews.csv": "tb_movies_reviews"
}

## Ingestão dos arquivos CSV

Os arquivos CSV são lidos sem inferência automática de schema, preservando os dados brutos da origem.

Nenhuma regra de limpeza ou transformação de negócio é aplicada nesta camada.

É adicionada apenas a coluna ingestion_datetime.

As tabelas são persistidas no formato Delta utilizando o modo Append.

In [0]:
from pyspark.sql.functions import current_timestamp


def ingest_csv_to_bronze(file_name, table_name):
    file_path = f"{landing_path}/{file_name}"
    table_path = f"{bronze_schema}.{table_name}"

    print(f"Ingerindo: {file_name}")
    print(f"Destino: {table_path}")

    # Leitura dos dados brutos sem inferência de tipos
    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "false")
        .csv(file_path)
    )

    # Registro do momento da ingestão
    df = df.withColumn(
        "ingestion_datetime",
        current_timestamp()
    )

    # Persistência da tabela em formato Delta
    (
        df.write
        .format("delta")
        .mode("append")
        .saveAsTable(table_path)
    )

    print(f"[OK] {table_path} ingerida com sucesso.")

In [0]:
for file_name, table_name in files_to_tables.items():
    ingest_csv_to_bronze(
        file_name,
        table_name
    )

Ingerindo: movies_info_TMDB_IMDB.csv
Destino: rocketlab.bronze.tb_movies_info
[OK] rocketlab.bronze.tb_movies_info ingerida com sucesso.
Ingerindo: movies_financials_IMDB_TMDB.csv
Destino: rocketlab.bronze.tb_movies_financials
[OK] rocketlab.bronze.tb_movies_financials ingerida com sucesso.
Ingerindo: movies_metrics_IMDB_TMDB.csv
Destino: rocketlab.bronze.tb_movies_metrics
[OK] rocketlab.bronze.tb_movies_metrics ingerida com sucesso.
Ingerindo: credits_and_tags_IMDB_TMDB.csv
Destino: rocketlab.bronze.tb_credits_and_tags
[OK] rocketlab.bronze.tb_credits_and_tags ingerida com sucesso.
Ingerindo: movies_reviews.csv
Destino: rocketlab.bronze.tb_movies_reviews
[OK] rocketlab.bronze.tb_movies_reviews ingerida com sucesso.


## Validação da ingestão

Após a ingestão, são verificadas:

- As tabelas criadas na camada Bronze.
- A quantidade de registros armazenados.
- A presença da coluna ingestion_datetime.

In [0]:
display(
    spark.sql(f"SHOW TABLES IN {bronze_schema}")
)

database,tableName,isTemporary
bronze,tb_credits_and_tags,false
bronze,tb_movies_financials,false
bronze,tb_movies_info,false
bronze,tb_movies_metrics,false
bronze,tb_movies_reviews,false


In [0]:
for table_name in files_to_tables.values():

    full_table_name = f"{bronze_schema}.{table_name}"

    df = spark.table(full_table_name)

    row_count = df.count()

    has_ingestion_datetime = (
        "ingestion_datetime" in df.columns
    )

    status = (
        "ok"
        if has_ingestion_datetime
        else "erro"
    )

    print(
        f"{status} {table_name} | "
        f"Registros: {row_count} | "
        f"ingestion_datetime: {has_ingestion_datetime}"
    )

ok tb_movies_info | Registros: 106930 | ingestion_datetime: True
ok tb_movies_financials | Registros: 106165 | ingestion_datetime: True
ok tb_movies_metrics | Registros: 107364 | ingestion_datetime: True
ok tb_credits_and_tags | Registros: 106320 | ingestion_datetime: True
ok tb_movies_reviews | Registros: 32412 | ingestion_datetime: True


In [0]:
# Validando se as tabelas são Deltas
for table_name in files_to_tables.values():
    full_table_name = f"{bronze_schema}.{table_name}"

    detail = spark.sql(
        f"DESCRIBE DETAIL {full_table_name}"
    ).first()

    print(
        f"ok {table_name} | "
        f"Formato: {detail['format']}"
    )

ok tb_movies_info | Formato: delta
ok tb_movies_financials | Formato: delta
ok tb_movies_metrics | Formato: delta
ok tb_credits_and_tags | Formato: delta
ok tb_movies_reviews | Formato: delta


## Ingestão da cotação do dólar

 Por padrão, é considerado um período de 7 dias corridos, pois a API não disponibiliza cotações em finais de semana e feriados.

Os dados obtidos serão armazenados na tabela bronze.tb_cotacao_dolar.

In [0]:
from datetime import datetime, timedelta

data_fim_default = datetime.now()
data_inicio_default = data_fim_default - timedelta(days=7)

data_inicio_default = data_inicio_default.strftime("%m-%d-%Y")
data_fim_default = data_fim_default.strftime("%m-%d-%Y")

print(f"Data inicial: {data_inicio_default}")
print(f"Data final: {data_fim_default}")

dbutils.widgets.text(
    "data_inicio",
    data_inicio_default,
    "Data inicial"
)

dbutils.widgets.text(
    "data_fim",
    data_fim_default,
    "Data final"
)

Data inicial: 09-13-2026
Data final: 09-20-2026


In [0]:
data_inicio = dbutils.widgets.get("data_inicio")
data_fim = dbutils.widgets.get("data_fim")

print(f"Período selecionado: {data_inicio} até {data_fim}")

# Validando as datas
def validar_data(data):
    try:
        datetime.strptime(data, "%m-%d-%Y")
        return True
    except ValueError:
        return False


if not validar_data(data_inicio):
    raise ValueError(
        "data_inicio deve estar no formato MM-DD-AAAA."
    )

if not validar_data(data_fim):
    raise ValueError(
        "data_fim deve estar no formato MM-DD-AAAA."
    )

print("ok datas válidas.")

inicio = datetime.strptime(data_inicio, "%m-%d-%Y")
fim = datetime.strptime(data_fim, "%m-%d-%Y")

if inicio > fim:
    raise ValueError(
        "A data inicial não pode ser posterior à data final."
    )

print("ok período válido.")

Período selecionado: 09-11-2026 até 09-18-2026
ok datas válidas.
ok período válido.


In [0]:
base_url = (
    "https://olinda.bcb.gov.br/olinda/servico/PTAX/"
    "versao/v1/odata/CotacaoDolarPeriodo"
)

url = (
    f"{base_url}"
    f"(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)"
    f"?@dataInicial='{data_inicio}'"
    f"&@dataFinalCotacao='{data_fim}'"
    f"&$select=dataHoraCotacao,cotacaoCompra"
    f"&$format=json"
)

print(url)

https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?@dataInicial='09-11-2026'&@dataFinalCotacao='09-18-2026'&$select=dataHoraCotacao,cotacaoCompra&$format=json


In [0]:
import requests

response = requests.get(
    url,
    timeout=30
)

response.raise_for_status()

print(f"Status HTTP: {response.status_code}")

response_json = response.json()

cotacoes = response_json.get("value", [])

print(f"Cotações retornadas: {len(cotacoes)}")

if not cotacoes:
    raise ValueError(
        "Nenhuma cotação foi encontrada para o período informado. "
        "Verifique se o intervalo inclui algum dia útil."
    )

cotacoes[:5]

Status HTTP: 200
Cotações retornadas: 6


[{'cotacaoCompra': 5.0912, 'dataHoraCotacao': '2026-09-11 13:07:22.532196'},
 {'cotacaoCompra': 5.169, 'dataHoraCotacao': '2026-09-14 13:10:08.144425'},
 {'cotacaoCompra': 5.1484, 'dataHoraCotacao': '2026-09-15 13:09:19.199664'},
 {'cotacaoCompra': 5.152, 'dataHoraCotacao': '2026-09-16 13:05:30.35873'},
 {'cotacaoCompra': 5.1515, 'dataHoraCotacao': '2026-09-17 13:03:21.858212'}]

In [0]:
df_cotacao_dolar = spark.createDataFrame(cotacoes)

df_cotacao_dolar.printSchema()
display(df_cotacao_dolar)

root
 |-- cotacaoCompra: double (nullable = true)
 |-- dataHoraCotacao: string (nullable = true)



cotacaoCompra,dataHoraCotacao
5.0912,2026-09-11 13:07:22.532196
5.169,2026-09-14 13:10:08.144425
5.1484,2026-09-15 13:09:19.199664
5.152,2026-09-16 13:05:30.35873
5.1515,2026-09-17 13:03:21.858212
5.1569,2026-09-18 13:03:34.742036


In [0]:
from pyspark.sql.functions import current_timestamp

df_cotacao_dolar = df_cotacao_dolar.withColumn(
    "ingestion_datetime",
    current_timestamp()
)

display(df_cotacao_dolar)

cotacaoCompra,dataHoraCotacao,ingestion_datetime
5.0912,2026-09-11 13:07:22.532196,2026-09-20T16:30:15.058Z
5.169,2026-09-14 13:10:08.144425,2026-09-20T16:30:15.058Z
5.1484,2026-09-15 13:09:19.199664,2026-09-20T16:30:15.058Z
5.152,2026-09-16 13:05:30.35873,2026-09-20T16:30:15.058Z
5.1515,2026-09-17 13:03:21.858212,2026-09-20T16:30:15.058Z
5.1569,2026-09-18 13:03:34.742036,2026-09-20T16:30:15.058Z


In [0]:
(
    df_cotacao_dolar.write
    .format("delta")
    .mode("append")
    .saveAsTable(f"{bronze_schema}.tb_cotacao_dolar")
)

print(
    f"ok -> {bronze_schema}.tb_cotacao_dolar "
    "ingerida com sucesso."
)

ok -> rocketlab.bronze.tb_cotacao_dolar ingerida com sucesso.


In [0]:
display(
    spark.table(f"{bronze_schema}.tb_cotacao_dolar")
)

df_cotacao_bronze = spark.table(
    f"{bronze_schema}.tb_cotacao_dolar"
)

print(
    f"Registros: {df_cotacao_bronze.count()}"
)

print(
    "ingestion_datetime:",
    "ingestion_datetime" in df_cotacao_bronze.columns
)

cotacaoCompra,dataHoraCotacao,ingestion_datetime
5.0912,2026-09-11 13:07:22.532196,2026-09-20T16:30:17.402Z
5.169,2026-09-14 13:10:08.144425,2026-09-20T16:30:17.402Z
5.1484,2026-09-15 13:09:19.199664,2026-09-20T16:30:17.402Z
5.152,2026-09-16 13:05:30.35873,2026-09-20T16:30:17.402Z
5.1515,2026-09-17 13:03:21.858212,2026-09-20T16:30:17.402Z
5.1569,2026-09-18 13:03:34.742036,2026-09-20T16:30:17.402Z


Registros: 6
ingestion_datetime: True


In [0]:
detail = spark.sql(
    f"DESCRIBE DETAIL {bronze_schema}.tb_cotacao_dolar"
).first()

print(
    f"Formato: {detail['format']}"
)

Formato: delta


## Validação final da camada Bronze

Nesta etapa faço a validação final das tabelas criadas na camada Bronze, verificando a quantidade de registros, a presença da coluna ingestion_datetime e o formato de armazenamento.

In [0]:
bronze_tables = [
    "tb_movies_info",
    "tb_movies_financials",
    "tb_movies_metrics",
    "tb_credits_and_tags",
    "tb_movies_reviews",
    "tb_cotacao_dolar"
]

for table_name in bronze_tables:

    full_table_name = f"{bronze_schema}.{table_name}"

    df = spark.table(full_table_name)

    row_count = df.count()

    has_ingestion_datetime = (
        "ingestion_datetime" in df.columns
    )

    detail = spark.sql(
        f"DESCRIBE DETAIL {full_table_name}"
    ).first()

    table_format = detail["format"]

    print(
        f"ok {table_name} | "
        f"Registros: {row_count} | "
        f"ingestion_datetime: {has_ingestion_datetime} | "
        f"Formato: {table_format}"
    )

display(
    spark.sql(f"SHOW TABLES IN {bronze_schema}")
)

ok tb_movies_info | Registros: 106930 | ingestion_datetime: True | Formato: delta
ok tb_movies_financials | Registros: 106165 | ingestion_datetime: True | Formato: delta
ok tb_movies_metrics | Registros: 107364 | ingestion_datetime: True | Formato: delta
ok tb_credits_and_tags | Registros: 106320 | ingestion_datetime: True | Formato: delta
ok tb_movies_reviews | Registros: 32412 | ingestion_datetime: True | Formato: delta
ok tb_cotacao_dolar | Registros: 6 | ingestion_datetime: True | Formato: delta


database,tableName,isTemporary
bronze,tb_cotacao_dolar,false
bronze,tb_credits_and_tags,false
bronze,tb_movies_financials,false
bronze,tb_movies_info,false
bronze,tb_movies_metrics,false
bronze,tb_movies_reviews,false
